## 1. Setup and Imports

In [21]:
!pip install -q sacrebleu

In [22]:
import os
import random
import pandas as pd
import sentencepiece as spm
import sacrebleu
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import zipfile
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.6.0+cu124
CUDA available: True


## 2. Configuration and Hyperparameters

In [23]:
# Data paths
TRAIN_SRC = "/kaggle/input/engtoviet-mt/train.csv"
VAL_SRC = "/kaggle/input/engtoviet-mt/val.csv"
TEST_SRC = "/kaggle/input/engtoviet-mt/test.csv"
SAVE_DIR = "./checkpoints"
SPM_ZH_PREFIX = os.path.join(SAVE_DIR, "spm_zh")
SPM_VI_PREFIX = os.path.join(SAVE_DIR, "spm_vi")

# Model hyperparameters
VOCAB_SIZE = 3000
EMB_SIZE = 64
HID_SIZE = 128
BATCH_SIZE = 128
EPOCHS = 40
LR = 0.005
MAX_LEN = 80
SEED = 42

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Set random seeds for reproducibility
os.makedirs(SAVE_DIR, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

Using device: cuda


## 3. Data Loading and Preprocessing

In [24]:
def read_lines(path):
    df = pd.read_csv(path, encoding='utf-8')
    return df['english'].astype(str), df['vietnamese'].astype(str)

# Load training and test data
train_src, train_tgt = read_lines(TRAIN_SRC)
valid_src, valid_tgt = read_lines(VAL_SRC)
test_src, test_tgt = read_lines(TEST_SRC)

print(f"Training samples: {len(train_src)}")
print(f"Validation samples: {len(valid_src)}")
print(f"Test samples: {len(test_src)}")
print(f"\nExample English sentence: {train_src[0]}")
print(f"Example Vietnamese sentence: {train_tgt[0]}")

Training samples: 108910
Validation samples: 13614
Test samples: 13614

Example English sentence: Your model of the world is what shapes you long term.
Example Vietnamese sentence: Mô hình thế giới của bạn là cái định hình bạn lâu dài.


In [25]:
valid_src = pd.concat([valid_src, test_src], ignore_index=True)
valid_tgt = pd.concat([valid_tgt, test_tgt], ignore_index=True)

print(f"Validation samples: {len(valid_src)}")

Validation samples: 27228


## 4. SentencePiece Tokenization

Train BPE tokenizers for both Chinese and Vietnamese.

In [26]:
def train_spm(input_file, model_prefix, vocab_size=VOCAB_SIZE):
    """Train a SentencePiece BPE model."""
    args = (
        f"--input={input_file} --model_prefix={model_prefix} --vocab_size={vocab_size} "
        "--model_type=bpe --character_coverage=1.0 "
        "--pad_id=0 --unk_id=1 --bos_id=2 --eos_id=3"
    )
    spm.SentencePieceTrainer.Train(args)
    print(f"Trained SentencePiece model: {model_prefix}.model")

def load_sp(model_path):
    """Load a trained SentencePiece model."""
    sp = spm.SentencePieceProcessor()
    sp.Load(model_path)
    return sp

In [27]:
# Train Chinese tokenizer
tmp_zh = os.path.join(SAVE_DIR, "tmp_zh.txt")
if not os.path.exists(SPM_ZH_PREFIX + ".model"):
    with open(tmp_zh, "w", encoding="utf-8") as f:
        for s in train_src:
            f.write(s + "\n")
    train_spm(tmp_zh, SPM_ZH_PREFIX)

# Train Vietnamese tokenizer
tmp_vi = os.path.join(SAVE_DIR, "tmp_vi.txt")
if not os.path.exists(SPM_VI_PREFIX + ".model"):
    with open(tmp_vi, "w", encoding="utf-8") as f:
        for s in train_tgt:
            f.write(s + "\n")
    train_spm(tmp_vi, SPM_VI_PREFIX)

# Load tokenizers
sp_zh = load_sp(SPM_ZH_PREFIX + ".model")
sp_vi = load_sp(SPM_VI_PREFIX + ".model")

print(f"\English vocab size: {sp_zh.GetPieceSize()}")
print(f"Vietnamese vocab size: {sp_vi.GetPieceSize()}")

# Test tokenization
test_sent = train_src[0]
tokens = sp_zh.EncodeAsIds(test_sent)
print(f"\nExample tokenization:")
print(f"Original: {test_sent}")
print(f"Token IDs: {tokens[:20]}...")

\English vocab size: 3000
Vietnamese vocab size: 3000

Example tokenization:
Original: Your model of the world is what shapes you long term.
Token IDs: [2848, 1153, 33, 9, 293, 58, 131, 1630, 29, 49, 633, 893, 2888, 2896]...


## 5. Dataset and DataLoader

In [28]:
class TranslationDataset(Dataset):
    """Dataset for Chinese-Vietnamese translation pairs."""
    
    def __init__(self, src, tgt, sp_src, sp_tgt, max_len=MAX_LEN):
        self.src = src
        self.tgt = tgt
        self.sp_src = sp_src
        self.sp_tgt = sp_tgt
        self.max_len = max_len

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        # Add BOS (2) and EOS (3) tokens
        src_ids = [2] + self.sp_src.EncodeAsIds(self.src[idx])[:self.max_len-2] + [3]
        tgt_ids = [2] + self.sp_tgt.EncodeAsIds(self.tgt[idx])[:self.max_len-2] + [3]
        return torch.tensor(src_ids), torch.tensor(tgt_ids)

def collate_fn(batch):
    """Collate function to pad sequences to the same length."""
    srcs, tgts = zip(*batch)
    max_src = max(len(s) for s in srcs)
    max_tgt = max(len(t) for t in tgts)
    
    # Pad with 0 (PAD token)
    src_pad = torch.zeros(len(batch), max_src, dtype=torch.long)
    tgt_pad = torch.zeros(len(batch), max_tgt, dtype=torch.long)
    
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src_pad[i, :len(s)] = s
        tgt_pad[i, :len(t)] = t
    
    return src_pad, tgt_pad

In [29]:
dataset = TranslationDataset(train_src, train_tgt, sp_zh, sp_vi)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=4, pin_memory=True)

valid_dataset = TranslationDataset(valid_src, valid_tgt, sp_zh, sp_vi)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f"Training batches: {len(dataloader)}")
print(f"Validation batches: {len(valid_loader)}")

Training batches: 851
Validation batches: 851


## 6. Model Architecture

### Encoder-Decoder with GRU

In [30]:
class EncoderRNN(nn.Module):
    """GRU-based encoder."""
    
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=0)
        self.dropout = nn.Dropout(0.3)
        self.rnn = nn.GRU(emb_size, hidden_size, batch_first=True)

    def forward(self, src):
        emb = self.dropout(self.embedding(src))
        _, hidden = self.rnn(emb)
        return hidden


class DecoderRNN(nn.Module):
    """GRU-based decoder with teacher forcing."""
    
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=0)
        self.dropout = nn.Dropout(0.3)
        self.rnn = nn.GRU(emb_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_step, hidden):
        emb = self.dropout(self.embedding(input_step))
        output, hidden = self.rnn(emb, hidden)
        pred = self.fc(output.squeeze(1))
        return pred, hidden


class Seq2Seq(nn.Module):
    """Sequence-to-sequence model."""
    
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.3):
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        vocab_size = self.decoder.fc.out_features
        
        # Store outputs
        outputs = torch.zeros(batch_size, tgt_len, vocab_size).to(src.device)
        
        # Encode source sentence
        hidden = self.encoder(src)
        
        # Start with BOS token
        input_step = tgt[:, 0].unsqueeze(1)
        
        # Decode step by step
        for t in range(1, tgt_len):
            pred, hidden = self.decoder(input_step, hidden)
            outputs[:, t] = pred
            
            # Teacher forcing
            teacher_force = random.random() < teacher_forcing_ratio
            input_step = tgt[:, t].unsqueeze(1) if teacher_force else pred.argmax(1).unsqueeze(1)
        
        return outputs

In [31]:
# Initialize model
encoder = EncoderRNN(sp_zh.GetPieceSize(), EMB_SIZE, HID_SIZE)
decoder = DecoderRNN(sp_vi.GetPieceSize(), EMB_SIZE, HID_SIZE)
model = Seq2Seq(encoder, decoder).to(DEVICE)

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model has {count_parameters(model):,} trainable parameters")
print(f"\nModel architecture:")
print(model)

Model has 919,992 trainable parameters

Model architecture:
Seq2Seq(
  (encoder): EncoderRNN(
    (embedding): Embedding(3000, 64, padding_idx=0)
    (dropout): Dropout(p=0.3, inplace=False)
    (rnn): GRU(64, 128, batch_first=True)
  )
  (decoder): DecoderRNN(
    (embedding): Embedding(3000, 64, padding_idx=0)
    (dropout): Dropout(p=0.3, inplace=False)
    (rnn): GRU(64, 128, batch_first=True)
    (fc): Linear(in_features=128, out_features=3000, bias=True)
  )
)


## 7. Training Functions

In [32]:
def train_epoch(model, dataloader, criterion, optimizer):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for src, tgt in dataloader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        
        optimizer.zero_grad()
        output = model(src, tgt)
        
        # Calculate loss (ignore first BOS token)
        loss = criterion(
            output[:, 1:].reshape(-1, output.size(-1)), 
            tgt[:, 1:].reshape(-1)
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate_bleu(model, dataloader, sp_tgt):
    """Evaluate model using SacreBLEU metric."""
    model.eval()
    hyps, refs = [], []
    
    pbar = tqdm(dataloader, desc="Evaluating", leave=False)
    for src, tgt in pbar:
        src = src.to(DEVICE)
        
        # Encode
        hidden = model.encoder(src)
        
        # Decode (greedy)
        input_step = torch.full((src.size(0), 1), 2, dtype=torch.long, device=DEVICE)
        decoded = [[] for _ in range(src.size(0))]
        
        for _ in range(MAX_LEN):
            pred, hidden = model.decoder(input_step, hidden)
            next_token = pred.argmax(1).unsqueeze(1)
            
            for i in range(src.size(0)):
                decoded[i].append(next_token[i].item())
            
            input_step = next_token
        
        # Convert to text
        for i in range(src.size(0)):
            ids = decoded[i]
            if 3 in ids:  # Stop at EOS
                ids = ids[:ids.index(3)]
            hyps.append(sp_tgt.DecodeIds(ids))
            
            ref_ids = tgt[i].tolist()[1:-1]  # Remove BOS and EOS
            refs.append(sp_tgt.DecodeIds([x for x in ref_ids if x not in [0, 1]]))
    
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    return bleu.score

## 8. Training Loop

In [33]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding
optimizer = optim.Adam(model.parameters(), lr=LR)

print("Starting training...\n")

# Training loop with best model saving
best_bleu = 0.0
best_model_path = os.path.join(SAVE_DIR, "best_model.pt")

for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(model, dataloader, criterion, optimizer)
    bleu = evaluate_bleu(model, valid_loader, sp_vi)
    
    # Save best model
    if bleu > best_bleu:
        best_bleu = bleu
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'bleu': bleu,
            'loss': loss
        }, best_model_path)
        print(f"Epoch {epoch:02d} | Loss={loss:.3f} | SacreBLEU={bleu:.2f} Best model saved!")
    else:
        print(f"Epoch {epoch:02d} | Loss={loss:.3f} | SacreBLEU={bleu:.2f}")

print(f"\nTraining completed!")
print(f"Best validation BLEU: {best_bleu:.2f}")
print(f"Best model saved to: {best_model_path}")

# Load best model for inference
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']}")

Starting training...



Epoch 01 | Loss=5.882 | SacreBLEU=1.36 Best model saved!


Epoch 02 | Loss=5.510 | SacreBLEU=1.65 Best model saved!


Epoch 03 | Loss=5.390 | SacreBLEU=1.79 Best model saved!


Epoch 04 | Loss=5.319 | SacreBLEU=1.85 Best model saved!


...

Epoch 36 | Loss=4.913 | SacreBLEU=2.88


Epoch 37 | Loss=4.923 | SacreBLEU=3.07 Best model saved!


Epoch 38 | Loss=4.911 | SacreBLEU=3.06


Epoch 39 | Loss=4.908 | SacreBLEU=2.96


Epoch 40 | Loss=4.912 | SacreBLEU=2.98

Training completed!
Best validation BLEU: 3.07
Best model saved to: ./checkpoints/best_model.pt
Loaded best model from epoch 37
